# Summary
The purpose of the following code is to verify bee behavior via visualization.

# Visualization of bee flight paths

## 1. Executing simulation
Execut one simulation and save the data.

In [ ]:
%matplotlib notebook

import pickle
import pandas as pd
import matplotlib.pyplot as plt
from model import OpenCoverModel, BlankModel
from tqdm import trange
import numpy as np
# ------------------------------
# User-defined simulation inputs
# ------------------------------

behavior = "my_behavior"  # Choose from: pure_levy, truncated_levy, composite_levy, biased_composite, my_behavior
seed_value = 0            # Random seed
steps = 10 * 60           # Number of simulation steps (e.g., 10 minutes)
n_agents = 60             # Number of bees
landing_prob = 1.0        # Bee landing probability

# ------------------------------
# Initialize and run the model
# ------------------------------

print(f"Running simulation for behavior: {behavior} with seed {seed_value}")

# 1. Initialize model
model = OpenCoverModel(
# model = BlankModel(
    n_agents=n_agents,
    bee_behavior=behavior,
    landing_prob=landing_prob,
    seed=seed_value
)

# 2. Step the model
# for _ in trange(steps, desc="Simulating"):
for _ in range(steps):
    model.step()

# ------------------------------
# Save model data (DataCollector results)
# ------------------------------

df = model.datacollector.get_agent_vars_dataframe()
output_filename = f"sim_results_{behavior}_seed_{seed_value}.pkl"

with open(output_filename, "wb") as f:
    pickle.dump(df, f)

print(f"Simulation complete. DataCollector results saved to {output_filename}")


In [ ]:
# Collecting sampled turning angles
all_angles = []
for agent in model.agents:
    if hasattr(agent, "turning_angles"):
        all_angles.extend(agent.turning_angles)

angle_filename = f"bee_turning_angles_{behavior}_seed_{seed_value}.pkl"
with open(angle_filename, "wb") as f:
    pickle.dump(all_angles, f)

print(f"Turning angles saved to {angle_filename}")

# Collecting sampled distances
all_distances = []
for agent in model.agents:
    if hasattr(agent, "jump_distances"):
        all_distances.extend(agent.jump_distances)
distance_filename = f"bee_jump_distances_{behavior}_seed_{seed_value}.pkl"
with open(distance_filename, "wb") as f:
    pickle.dump(all_distances, f)

print(f"Jumping distance saved to {distance_filename}")

### Turning angles verification

In [ ]:
%matplotlib inline
# Define your custom PDF
x_discrete = np.array([-180, -135, -90, -45, 0, 45, 90, 135, 180])
y_discrete = np.array([0.06, 0.07, 0.13, 0.17, 0.20, 0.17, 0.13, 0.07, 0.06])

x_fine = np.linspace(-180, 180, 1000)
pdf_fine = np.interp(x_fine, x_discrete, y_discrete)
dx_pdf = x_fine[1] - x_fine[0]
pdf_fine /= np.sum(pdf_fine * dx_pdf)

# Plot histogram
n, bins, patches = plt.hist(
    all_angles,
    bins=36,
    range=(-180, 180),
    density=True,
    alpha=0.5,
    color="orange"
)
patches[0].set_label("Sampled Angles")  # Set label only once

# Plot target PDF
plt.plot(
    x_fine,
    pdf_fine,
    label="Biased Turning Angles",
    color="blue",
    linewidth=2
)

plt.title("Sampled vs Target PDF of Free-Flight Turning Angles")
plt.xlabel("Turning Angle (degrees)")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

### Sampling Distance Verification

In [ ]:
%matplotlib inline
# Parameters from your model
alpha = 1.5
min_distance = 0.7
max_distance = max(all_distances) + 1  # To fit plot range

# Generate x values for PDF
x_vals = np.linspace(min_distance, max_distance, 1000)

# Compute Lévy PDF (unnormalized), then normalize
levy_pdf = x_vals ** -(1 + alpha)
dx = x_vals[1] - x_vals[0]
levy_pdf /= np.sum(levy_pdf * dx)  # Normalize to make it a valid PDF

# Plot
%matplotlib inline
if all_distances:
    plt.hist(
        all_distances,
        bins=50,
        alpha=0.7,
        density=True,
        label="Sampled Distances",
        color="green"
    )
    plt.plot(x_vals, levy_pdf, label=f"Theoretical Lévy PDF (α={alpha})", color='blue', linewidth=2)
    plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Long jump threshold (x = 3)')

    plt.xlim(0, max_distance)
    plt.title("Distribution of Bee Jump Distances (Truncated at x=20)")
    plt.xlabel("Jump Distance (meters)")
    plt.ylabel("Density")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("No jump distances recorded — skipping plot.")

## 2. Visualize flight path

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

In [ ]:
%matplotlib inline
def load_dataframe(filename):
    """Loads the dataframe from a pickle file."""
    with open(filename, "rb") as f:
        results = pickle.load(f)
    df = pd.DataFrame(results)
    return df.reset_index()


import matplotlib.pyplot as plt
import numpy as np

def load_dataframe(filename):
    """Loads the dataframe from a pickle file."""
    with open(filename, "rb") as f:
        results = pickle.load(f)
    df = pd.DataFrame(results)
    return df.reset_index()


def vis_bees_path(df, bee_ids, start_time_str="09:00", duration_minutes=10, x_buffer=5):
    """
    Plot flight paths of selected bees with direction arrows,
    including the first beehive and relevant plant locations.

    Parameters:
        df (pd.DataFrame): Simulation dataframe.
        bee_ids (list of int): List of bee AgentIDs to visualize.
        start_time_str (str): Start time in HH:MM format (24h clock starting from 09:00).
        duration_minutes (int): Duration in minutes to plot.
        x_buffer (float): Extra range added to max x for plant inclusion.
        
    NOTE: Choose bees from beehive 1 (beeids 2 to 62). Need to add extra data "from hive" in model.py
    """
    def time_str_to_step(time_str):
        hh, mm = map(int, time_str.split(":"))
        return (hh - 9) * 3600 + mm * 60

    start_step = time_str_to_step(start_time_str)
    end_step = start_step + duration_minutes * 60

    # Filter bees
    bee_df = df[(df["Agent Type"] == "bee") &
                (df["Step"] >= start_step) &
                (df["Step"] < end_step) &
                (df["AgentID"].isin(bee_ids))].copy()
    
    if bee_df.empty:
        print("No bee data found for the given IDs and time window.")
        return

    # Extract bee positions
    bee_df[['x', 'y']] = bee_df['Bee Positions'].apply(pd.Series)

    # Get x-range from bee paths
    min_x = bee_df['x'].min()
    max_x = bee_df['x'].max()

    # Prepare figure
    plt.figure(figsize=(10, 6))
    cmap = plt.colormaps.get_cmap("tab10").resampled(len(bee_ids))

    # Plot bee flight paths
    for idx, bee_id in enumerate(bee_ids):
        track = bee_df[bee_df['AgentID'] == bee_id].sort_values(by="Step")
        x = track['x'].values
        y = track['y'].values
        u = x[1:] - x[:-1]
        v = y[1:] - y[:-1]

        plt.plot(x, y, label=f"Bee {bee_id}", color=cmap(idx), linewidth=1.5, alpha=0.7)
        plt.quiver(x[:-1], y[:-1], u, v, angles='xy', scale_units='xy', scale=1,
                   width=0.0025, headwidth=3, headlength=4, color=cmap(idx))

    # Plot the first beehive at Step 0
    beehive_df = df[(df["Agent Type"] == "beehive") & (df["Step"] == 0)].copy()
    if not beehive_df.empty:
        beehive_df[['x', 'y']] = beehive_df['Bee Positions'].apply(pd.Series)
        beehive_df = beehive_df.sort_values(by="unique_id").head(1)  # First hive only
        for _, hive in beehive_df.iterrows():
            plt.scatter(hive['x'], hive['y'], marker='^', c='brown', edgecolors='black', s=100, zorder=3)
            plt.text(hive['x'], hive['y'] + 0.5, f"Hive {hive['unique_id']}",
                     ha='center', va='bottom', fontsize=9, color='black')

    # Plot filtered plant locations at Step 0
    plant_df = df[(df["Agent Type"] == "plant") & (df["Step"] == 0)].copy()
    if not plant_df.empty:
        plant_df[['x', 'y']] = plant_df['Bee Positions'].apply(pd.Series)
        filtered_plants = plant_df[(plant_df['x'] >= min_x) & (plant_df['x'] <= max_x + x_buffer)]
        plt.scatter(filtered_plants['x'], filtered_plants['y'], marker='o', c='green', edgecolors='black', s=40, alpha=0.6, label='Plant', zorder=1)

    # Final formatting
    plt.title(f"Bee Flight Paths from {start_time_str} for {duration_minutes} min")
    plt.xlabel("X Position")
    plt.ylabel("Y Position")
    plt.grid(True)
    plt.axis("equal")
    plt.legend()
    plt.show()


In [ ]:
df = load_dataframe("sim_results_my_behavior_seed_0.pkl")

In [ ]:
%matplotlib inline
# Select bee IDs to visualize
vis_bees_path(df, bee_ids=[5], start_time_str="9:00", duration_minutes=10)

# Visualisation of specific sections of simulation

In [ ]:
import pickle
import pandas as pd

def load_run_dataframe(pkl_filename):
    """
    Loads a simulation pickle file (single-run or batch) and returns a DataFrame
    with MultiIndex (Step, unique_id).
    
    If Step and unique_id are already part of the index, leaves them as-is.
    """
    with open(pkl_filename, "rb") as f:
        results = pickle.load(f)

    # Case 1: It's a list of records → batch or single run
    if isinstance(results, list):
        df_run = pd.DataFrame(results)
        if "Step" in df_run.columns and "unique_id" in df_run.columns:
            df_run.set_index(["Step", "unique_id"], inplace=True)
        else:
            raise KeyError("Expected 'Step' and 'unique_id' in DataFrame columns.")
    # Case 2: Already a DataFrame
    elif isinstance(results, pd.DataFrame):
        df_run = results.copy()
        if not isinstance(df_run.index, pd.MultiIndex):
            if "Step" in df_run.columns and "unique_id" in df_run.columns:
                df_run.set_index(["Step", "unique_id"], inplace=True)
            else:
                raise KeyError("Expected 'Step' and 'unique_id' in DataFrame columns.")
    else:
        raise TypeError("Unsupported pickle format. Expected list or DataFrame.")

    # Diagnostics
    print(f"Loaded file '{pkl_filename}' with {len(df_run)} rows")
    print(f"Step range: {df_run.index.get_level_values('Step').min()} to {df_run.index.get_level_values('Step').max()}")
    print(f"Agent types: {df_run['Agent Type'].unique()}")

    return df_run


In [ ]:
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# import matplotlib as mpl

# def visualize_simulation_partial(df, start_time_str="9:00", duration_minutes=10, dynamic_plants=True, frame_skip=1, highlight_bee_ids=None):
#     mpl.rcParams['animation.embed_limit'] = 50

#     bee_df = df[df["Agent Type"] == "bee"]
#     plant_df = df[df["Agent Type"] == "plant"]
#     beehive_df = df[df["Agent Type"] == "beehive"]

#     steps = sorted(bee_df.index.get_level_values("Step").unique())

#     def time_str_to_seconds(time_str):
#         hh, mm = map(int, time_str.split(":"))
#         return (hh - 9) * 3600 + mm * 60

#     def step_to_time_str(step):
#         hh = 9 + step // 3600
#         mm = (step % 3600) // 60
#         ss = step % 60
#         return f"{hh:02d}:{mm:02d}:{ss:02d}"

#     start_second = time_str_to_seconds(start_time_str)
#     end_second = start_second + duration_minutes * 60
#     selected_steps = [step for step in steps if start_second <= step < end_second]

#     if len(selected_steps) == 0:
#         print("Warning: No steps found in the specified time interval.")
#         return None, None

#     x_min, x_max = 0, 50
#     y_min, y_max = 0, 2.85 * 4

#     fig, ax = plt.subplots(figsize=(x_max/2, (y_max-y_min)/2))
#     ax.set_facecolor("gray")

#     for x0, y0, x1, y1 in [
#         (x_min, y_min, x_max, y_min),
#         (x_min, y_max, x_max, y_max),
#         (x_min, y_min, x_min, y_max),
#         (x_max, y_min, x_max, y_max),
#     ]:
#         ax.plot([x0, x1], [y0, y1], color="black", linewidth=1.5)

#     ax.set_xlim(x_min, x_max)
#     ax.set_ylim(y_min, y_max)
#     ax.set_xlabel("X Position")
#     ax.set_ylabel("Y Position")
#     ax.set_title("Bee Movement Relative to Plants and Beehives (X ≤ 50)")

#     plant_style_df = plant_df.loc[0].reset_index().set_index("unique_id")
#     bee_style_df = bee_df.loc[0].reset_index().set_index("unique_id")
#     beehive_df_static = beehive_df.loc[0].reset_index()

#     for _, hive in beehive_df_static.iterrows():
#         pos = hive["Bee Positions"]
#         style = hive["Plot Code"]
#         if pos[0] <= x_max:
#             ax.scatter(pos[0], pos[1],
#                        marker=style.get("marker", "^"),
#                        c=style.get("color", "brown"),
#                        edgecolors=style.get("edgecolors", "none"),
#                        linewidths=style.get("linewidths", 0.8),
#                        s=80, zorder=0)
#             ax.text(pos[0], pos[1] + 0.5, f"Hive {hive['unique_id']}",
#                     fontsize=8, ha='center', va='bottom', color='white', zorder=5)

#     bee_scatters, plant_scatters = {}, {}
#     bee_texts, bee_coord_texts, bee_timer_texts = {}, {}, {}
#     plant_nectar_texts, plant_scent_texts = {}, {}

#     for uid, row in plant_style_df.iterrows():
#         pos = row["Bee Positions"]
#         if pos[0] <= x_max:
#             style = row["Plot Code"]
#             scat = ax.scatter([], [], marker=style.get("marker", "o"),
#                               c=style.get("color"),
#                               edgecolors=style.get("edgecolors", "black"),
#                               linewidths=style.get("linewidths", 0.8),
#                               s=40, zorder=1)
#             plant_scatters[uid] = scat
#             nectar_text = ax.text(pos[0], pos[1] - 0.5, "", fontsize=8, color="blue", ha='center', va='top', zorder=2)
#             scent_text = ax.text(pos[0], pos[1] - 0.8, "", fontsize=8, color="purple", ha='center', va='top', zorder=2)
#             plant_nectar_texts[uid] = nectar_text
#             plant_scent_texts[uid] = scent_text

#     for uid, row in bee_style_df.iterrows():
#         style = row["Plot Code"]
#         color = "red" if highlight_bee_ids and uid in highlight_bee_ids else style.get("color")
#         scat = ax.scatter([], [], marker=style.get("marker", "o"),
#                           c=color,
#                           edgecolors=style.get("edgecolors", "black"),
#                           linewidths=style.get("linewidths", 0.8),
#                           s=40, zorder=2)
#         bee_scatters[uid] = scat
#         if highlight_bee_ids and uid in highlight_bee_ids:
#             text_id = ax.text(0, 0, str(uid), fontsize=8, color="red", ha='center', va='bottom', zorder=3)
#             text_coord = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
#             text_timer = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
#             bee_texts[uid] = text_id
#             bee_coord_texts[uid] = text_coord
#             bee_timer_texts[uid] = text_timer

#     def update(frame):
#         current_bees = bee_df.loc[frame]
#         if isinstance(current_bees, pd.Series):
#             current_bees = pd.DataFrame([current_bees])
#         current_bees = current_bees.reset_index()
#         for _, bee in current_bees.iterrows():
#             uid, pos = bee["unique_id"], bee["Bee Positions"]
#             x, y = pos[0], pos[1]
#             forage_timer = bee.get("forage_timer", bee.get("Forage Timer", None))
#             if uid in bee_scatters:
#                 bee_scatters[uid].set_offsets([pos] if x <= x_max else [[np.nan, np.nan]])
#             if uid in bee_texts:
#                 if x <= x_max:
#                     bee_texts[uid].set_position((x, y + 1.0))
#                     bee_coord_texts[uid].set_position((x, y + 0.65))
#                     bee_coord_texts[uid].set_text(f"({x:.1f}, {y:.1f})")
#                     bee_timer_texts[uid].set_position((x, y + 0.35))
#                     bee_timer_texts[uid].set_text(f"{forage_timer}" if forage_timer is not None else "")
#                 else:
#                     for d in (bee_texts, bee_coord_texts, bee_timer_texts):
#                         d[uid].set_position((np.nan, np.nan))
#                         d[uid].set_text("")

#         if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
#             current_plants = plant_df.loc[frame]
#             if isinstance(current_plants, pd.Series):
#                 current_plants = pd.DataFrame([current_plants])
#             current_plants = current_plants.reset_index()
#             for _, plant in current_plants.iterrows():
#                 uid, pos = plant["unique_id"], plant["Bee Positions"]
#                 x, y = pos[0], pos[1]
#                 nectar, scent = plant.get("Plant nectar amount", None), plant.get("Plant scent timer", None)
#                 if uid in plant_scatters:
#                     if x <= x_max:
#                         plant_scatters[uid].set_offsets([pos])
#                         plant_nectar_texts[uid].set_position((x, y - 0.5))
#                         plant_nectar_texts[uid].set_text(f"{nectar:.1f}" if nectar is not None else "")
#                         plant_scent_texts[uid].set_position((x, y - 0.8))
#                         plant_scent_texts[uid].set_text(f"{scent}" if scent is not None else "")
#                     else:
#                         plant_scatters[uid].set_offsets([[np.nan, np.nan]])
#                         plant_nectar_texts[uid].set_position((np.nan, np.nan))
#                         plant_nectar_texts[uid].set_text("")
#                         plant_scent_texts[uid].set_position((np.nan, np.nan))
#                         plant_scent_texts[uid].set_text("")

#         ax.set_title(f"Step {frame} ({step_to_time_str(frame)}) - Bee Movement Relative to Plants and Beehives (X ≤ 50)")
#         return list(bee_scatters.values()) + list(plant_scatters.values()) + \
#                list(bee_texts.values()) + list(bee_coord_texts.values()) + list(bee_timer_texts.values()) + \
#                list(plant_nectar_texts.values()) + list(plant_scent_texts.values())

#     frame_gen = (i for i in tqdm(selected_steps[::frame_skip], desc="Rendering animation"))
#     anim = FuncAnimation(fig, update, frames=frame_gen, blit=False, interval=300, repeat=False)

#     return anim, HTML(anim.to_jshtml())


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib as mpl

def visualize_simulation_partial(df, start_time_str="9:00", duration_minutes=10, dynamic_plants=True, frame_skip=1, highlight_bee_ids=None):
    mpl.rcParams['animation.embed_limit'] = 50

    bee_df = df[df["Agent Type"] == "bee"]
    plant_df = df[df["Agent Type"] == "plant"]
    beehive_df = df[df["Agent Type"] == "beehive"]

    steps = sorted(bee_df.index.get_level_values("Step").unique())

    def time_str_to_seconds(time_str):
        hh, mm = map(int, time_str.split(":"))
        return (hh - 9) * 3600 + mm * 60

    def step_to_time_str(step):
        hh = 9 + step // 3600
        mm = (step % 3600) // 60
        ss = step % 60
        return f"{hh:02d}:{mm:02d}:{ss:02d}"

    start_second = time_str_to_seconds(start_time_str)
    end_second = start_second + duration_minutes * 60
    selected_steps = [step for step in steps if start_second <= step < end_second]

    if len(selected_steps) == 0:
        print("Warning: No steps found in the specified time interval.")
        return None, None

    x_min, x_max = 0, 50
    y_min, y_max = 0, 2.85 * 4

    fig, ax = plt.subplots(figsize=(x_max/2, (y_max-y_min)/2))
    ax.set_facecolor("gray")

    for x0, y0, x1, y1 in [
        (x_min, y_min, x_max, y_min),
        (x_min, y_max, x_max, y_max),
        (x_min, y_min, x_min, y_max),
        (x_max, y_min, x_max, y_max),
    ]:
        ax.plot([x0, x1], [y0, y1], color="black", linewidth=1.5)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Bee Movement Relative to Plants and Beehives (X ≤ 50)")

    plant_style_df = plant_df.loc[0].reset_index().set_index("unique_id")
    bee_style_df = bee_df.loc[0].reset_index().set_index("unique_id")
    beehive_df_static = beehive_df.loc[0].reset_index()

    for _, hive in beehive_df_static.iterrows():
        pos = hive["Bee Positions"]
        style = hive["Plot Code"]
        if pos[0] <= x_max:
            ax.scatter(pos[0], pos[1],
                       marker=style.get("marker", "^"),
                       c=style.get("color", "brown"),
                       edgecolors=style.get("edgecolors", "none"),
                       linewidths=style.get("linewidths", 0.8),
                       s=80, zorder=0)
            ax.text(pos[0], pos[1] + 0.5, f"Hive {hive['unique_id']}",
                    fontsize=8, ha='center', va='bottom', color='white', zorder=5)

    bee_scatters, plant_scatters = {}, {}
    bee_texts, bee_coord_texts, bee_timer_texts = {}, {}, {}
    plant_nectar_texts, plant_scent_texts = {}, {}

    bee_latest_visited_texts = {}

    for uid, row in plant_style_df.iterrows():
        pos = row["Bee Positions"]
        if pos[0] <= x_max:
            style = row["Plot Code"]
            scat = ax.scatter([], [], marker=style.get("marker", "o"),
                              c=style.get("color"),
                              edgecolors=style.get("edgecolors", "black"),
                              linewidths=style.get("linewidths", 0.8),
                              s=40, zorder=1)
            plant_scatters[uid] = scat
            nectar_text = ax.text(pos[0], pos[1] - 0.5, "", fontsize=8, color="blue", ha='center', va='top', zorder=2)
            scent_text = ax.text(pos[0], pos[1] - 0.8, "", fontsize=8, color="purple", ha='center', va='top', zorder=2)
            plant_nectar_texts[uid] = nectar_text
            plant_scent_texts[uid] = scent_text

    for uid, row in bee_style_df.iterrows():
        style = row["Plot Code"]
        color = "red" if highlight_bee_ids and uid in highlight_bee_ids else style.get("color")
        scat = ax.scatter([], [], marker=style.get("marker", "o"),
                          c=color,
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=2)
        bee_scatters[uid] = scat
        if highlight_bee_ids and uid in highlight_bee_ids:
            text_id = ax.text(0, 0, str(uid), fontsize=8, color="red", ha='center', va='bottom', zorder=3)
            text_coord = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            text_timer = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            bee_texts[uid] = text_id
            bee_coord_texts[uid] = text_coord
            bee_timer_texts[uid] = text_timer
            text_latest = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            bee_latest_visited_texts[uid] = text_latest

    def update(frame):
        current_bees = bee_df.loc[frame]
        if isinstance(current_bees, pd.Series):
            current_bees = pd.DataFrame([current_bees])
        current_bees = current_bees.reset_index()
        for _, bee in current_bees.iterrows():
            uid, pos = bee["unique_id"], bee["Bee Positions"]
            x, y = pos[0], pos[1]
            forage_timer = bee.get("forage_timer", bee.get("Forage Timer", None))
            if uid in bee_scatters:
                bee_scatters[uid].set_offsets([pos] if x <= x_max else [[np.nan, np.nan]])
            if uid in bee_texts:
                if x <= x_max:
                    # Move all labels higher above the bee
                    bee_texts[uid].set_position((x, y + 2.0))          # Bee ID (highest)
                    bee_coord_texts[uid].set_position((x, y + 1.65))    # Coordinates
                    bee_coord_texts[uid].set_text(f"({x:.1f}, {y:.1f})")
                    bee_timer_texts[uid].set_position((x, y + 1.35))    # Forage timer
                    bee_timer_texts[uid].set_text(f"{forage_timer}" if forage_timer is not None else "")
                    latest_visited = bee.get("Latest Visited Plant Location", [])
                    formatted = [f"({p[0]:.1f}, {p[1]:.1f})" for p in latest_visited] if isinstance(latest_visited, list) else []
                    bee_latest_visited_texts[uid].set_position((x, y + 1.08)) # Latest visited, lowest
                    bee_latest_visited_texts[uid].set_text(f"Visited: {formatted}")
                else:
                    for d in (bee_texts, bee_coord_texts, bee_timer_texts, bee_latest_visited_texts):
                        d[uid].set_position((np.nan, np.nan))
                        d[uid].set_text("")

        if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
            current_plants = plant_df.loc[frame]
            if isinstance(current_plants, pd.Series):
                current_plants = pd.DataFrame([current_plants])
            current_plants = current_plants.reset_index()
            for _, plant in current_plants.iterrows():
                uid, pos = plant["unique_id"], plant["Bee Positions"]
                x, y = pos[0], pos[1]
                nectar, scent = plant.get("Plant nectar amount", None), plant.get("Plant scent timer", None)
                if uid in plant_scatters:
                    if x <= x_max:
                        plant_scatters[uid].set_offsets([pos])
                        plant_nectar_texts[uid].set_position((x, y - 0.5))
                        plant_nectar_texts[uid].set_text(f"{nectar:.1f}" if nectar is not None else "")
                        plant_scent_texts[uid].set_position((x, y - 0.8))
                        plant_scent_texts[uid].set_text(f"{scent}" if scent is not None else "")
                    else:
                        plant_scatters[uid].set_offsets([[np.nan, np.nan]])
                        plant_nectar_texts[uid].set_position((np.nan, np.nan))
                        plant_nectar_texts[uid].set_text("")
                        plant_scent_texts[uid].set_position((np.nan, np.nan))
                        plant_scent_texts[uid].set_text("")

        ax.set_title(f"Step {frame} ({step_to_time_str(frame)}) - Bee Movement Relative to Plants and Beehives (X ≤ 50)")
        return (
            list(bee_scatters.values()) +
            list(plant_scatters.values()) +
            list(bee_texts.values()) +
            list(bee_coord_texts.values()) +
            list(bee_timer_texts.values()) +
            list(bee_latest_visited_texts.values()) +
            list(plant_nectar_texts.values()) +
            list(plant_scent_texts.values())
        )

    frame_gen = (i for i in tqdm(selected_steps[::frame_skip], desc="Rendering animation"))
    anim = FuncAnimation(fig, update, frames=frame_gen, blit=False, interval=300, repeat=False)

    return anim, HTML(anim.to_jshtml())


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib as mpl

def visualize_simulation_partial(df, start_time_str="9:00", duration_minutes=10, dynamic_plants=True, frame_skip=1, highlight_bee_ids=None):
    mpl.rcParams['animation.embed_limit'] = 50

    bee_df = df[df["Agent Type"] == "bee"]
    plant_df = df[df["Agent Type"] == "plant"]
    beehive_df = df[df["Agent Type"] == "beehive"]

    steps = sorted(bee_df.index.get_level_values("Step").unique())

    def time_str_to_seconds(time_str):
        hh, mm = map(int, time_str.split(":"))
        return (hh - 9) * 3600 + mm * 60

    def step_to_time_str(step):
        hh = 9 + step // 3600
        mm = (step % 3600) // 60
        ss = step % 60
        return f"{hh:02d}:{mm:02d}:{ss:02d}"

    start_second = time_str_to_seconds(start_time_str)
    end_second = start_second + duration_minutes * 60
    selected_steps = [step for step in steps if start_second <= step < end_second]

    if len(selected_steps) == 0:
        print("Warning: No steps found in the specified time interval.")
        return None, None

    x_min, x_max = 0, 50
    y_min, y_max = 0, 2.85 * 4

    fig, ax = plt.subplots(figsize=(x_max/2, (y_max-y_min)/2))
    ax.set_facecolor("gray")

    for x0, y0, x1, y1 in [
        (x_min, y_min, x_max, y_min),
        (x_min, y_max, x_max, y_max),
        (x_min, y_min, x_min, y_max),
        (x_max, y_min, x_max, y_max),
    ]:
        ax.plot([x0, x1], [y0, y1], color="black", linewidth=1.5)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Bee Movement Relative to Plants and Beehives (X ≤ 50)")

    plant_style_df = plant_df.loc[0].reset_index().set_index("unique_id")
    bee_style_df = bee_df.loc[0].reset_index().set_index("unique_id")
    beehive_df_static = beehive_df.loc[0].reset_index()

    for _, hive in beehive_df_static.iterrows():
        pos = hive["Bee Positions"]
        style = hive["Plot Code"]
        if pos[0] <= x_max:
            ax.scatter(pos[0], pos[1],
                       marker=style.get("marker", "^"),
                       c=style.get("color", "brown"),
                       edgecolors=style.get("edgecolors", "none"),
                       linewidths=style.get("linewidths", 0.8),
                       s=80, zorder=0)
            ax.text(pos[0], pos[1] + 0.5, f"Hive {hive['unique_id']}",
                    fontsize=8, ha='center', va='bottom', color='white', zorder=5)

    bee_scatters, plant_scatters = {}, {}
    bee_texts, bee_coord_texts, bee_timer_texts = {}, {}, {}
    plant_nectar_texts, plant_scent_texts = {}, {}
    bee_latest_visited_texts = {}
    bee_white_id_texts = {}

    for uid, row in plant_style_df.iterrows():
        pos = row["Bee Positions"]
        if pos[0] <= x_max:
            style = row["Plot Code"]
            scat = ax.scatter([], [], marker=style.get("marker", "o"),
                              c=style.get("color"),
                              edgecolors=style.get("edgecolors", "black"),
                              linewidths=style.get("linewidths", 0.8),
                              s=40, zorder=1)
            plant_scatters[uid] = scat
            nectar_text = ax.text(pos[0], pos[1] - 0.5, "", fontsize=8, color="blue", ha='center', va='top', zorder=2)
            scent_text = ax.text(pos[0], pos[1] - 0.8, "", fontsize=8, color="purple", ha='center', va='top', zorder=2)
            plant_nectar_texts[uid] = nectar_text
            plant_scent_texts[uid] = scent_text

    for uid, row in bee_style_df.iterrows():
        style = row["Plot Code"]
        color = "red" if highlight_bee_ids and uid in highlight_bee_ids else style.get("color")
        scat = ax.scatter([], [], marker=style.get("marker", "o"),
                          c=color,
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=2)
        bee_scatters[uid] = scat
        # Red (highlighted) bees: only draw the original above-bee red labels
        if highlight_bee_ids and uid in highlight_bee_ids:
            text_id = ax.text(0, 0, str(uid), fontsize=8, color="red", ha='center', va='bottom', zorder=3)
            text_coord = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            text_timer = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            bee_texts[uid] = text_id
            bee_coord_texts[uid] = text_coord
            bee_timer_texts[uid] = text_timer
            text_latest = ax.text(0, 0, "", fontsize=8, color="red", ha='center', va='top', zorder=3)
            bee_latest_visited_texts[uid] = text_latest
        # Only for white bees NOT highlighted, draw the bee_id below the bee (in white)
        if style.get("color") == "white" and not (highlight_bee_ids and uid in highlight_bee_ids):
            text_white = ax.text(0, 0, str(uid), fontsize=8, color="white", ha='center', va='top', zorder=3)
            bee_white_id_texts[uid] = text_white

    def update(frame):
        current_bees = bee_df.loc[frame]
        if isinstance(current_bees, pd.Series):
            current_bees = pd.DataFrame([current_bees])
        current_bees = current_bees.reset_index()
        for _, bee in current_bees.iterrows():
            uid, pos = bee["unique_id"], bee["Bee Positions"]
            x, y = pos[0], pos[1]
            forage_timer = bee.get("forage_timer", bee.get("Forage Timer", None))
            if uid in bee_scatters:
                bee_scatters[uid].set_offsets([pos] if x <= x_max else [[np.nan, np.nan]])
            # Highlighted/red bees: original above-bee labels (not the below id)
            if uid in bee_texts:
                if x <= x_max:
                    bee_texts[uid].set_position((x, y + 2.0))
                    bee_coord_texts[uid].set_position((x, y + 1.65))
                    bee_coord_texts[uid].set_text(f"({x:.2f}, {y:.2f})")
                    bee_timer_texts[uid].set_position((x, y + 1.35))
                    bee_timer_texts[uid].set_text(f"{forage_timer}" if forage_timer is not None else "")
                    latest_visited = bee.get("Latest Visited Plant Location", [])
                    formatted = [f"({p[0]:.2f}, {p[1]:.2f})" for p in latest_visited] if isinstance(latest_visited, list) else []
                    bee_latest_visited_texts[uid].set_position((x, y + 1.08))
                    bee_latest_visited_texts[uid].set_text(f"Visited: {formatted}")
                else:
                    for d in (bee_texts, bee_coord_texts, bee_timer_texts, bee_latest_visited_texts):
                        d[uid].set_position((np.nan, np.nan))
                        d[uid].set_text("")
            # White bees not highlighted: show bee_id below (in white)
            if uid in bee_white_id_texts:
                if x <= x_max:
                    bee_white_id_texts[uid].set_position((x, y - 0.5))
                    bee_white_id_texts[uid].set_text(str(uid))
                else:
                    bee_white_id_texts[uid].set_position((np.nan, np.nan))
                    bee_white_id_texts[uid].set_text("")

        if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
            current_plants = plant_df.loc[frame]
            if isinstance(current_plants, pd.Series):
                current_plants = pd.DataFrame([current_plants])
            current_plants = current_plants.reset_index()
            for _, plant in current_plants.iterrows():
                uid, pos = plant["unique_id"], plant["Bee Positions"]
                x, y = pos[0], pos[1]
                nectar, scent = plant.get("Plant nectar amount", None), plant.get("Plant scent timer", None)
                if uid in plant_scatters:
                    if x <= x_max:
                        plant_scatters[uid].set_offsets([pos])
                        plant_nectar_texts[uid].set_position((x, y - 0.5))
                        plant_nectar_texts[uid].set_text(f"{nectar:.1f}" if nectar is not None else "")
                        plant_scent_texts[uid].set_position((x, y - 0.8))
                        plant_scent_texts[uid].set_text(f"{scent}" if scent is not None else "")
                    else:
                        plant_scatters[uid].set_offsets([[np.nan, np.nan]])
                        plant_nectar_texts[uid].set_position((np.nan, np.nan))
                        plant_nectar_texts[uid].set_text("")
                        plant_scent_texts[uid].set_position((np.nan, np.nan))
                        plant_scent_texts[uid].set_text("")

        ax.set_title(f"Step {frame} ({step_to_time_str(frame)}) - Bee Movement Relative to Plants and Beehives (X ≤ 50)")
        return (
            list(bee_scatters.values()) +
            list(plant_scatters.values()) +
            list(bee_texts.values()) +
            list(bee_coord_texts.values()) +
            list(bee_timer_texts.values()) +
            list(bee_latest_visited_texts.values()) +
            list(bee_white_id_texts.values()) +      # show white bee IDs only (not highlighted)
            list(plant_nectar_texts.values()) +
            list(plant_scent_texts.values())
        )

    frame_gen = (i for i in tqdm(selected_steps[::frame_skip], desc="Rendering animation"))
    anim = FuncAnimation(fig, update, frames=frame_gen, blit=False, interval=300, repeat=False)

    return anim, HTML(anim.to_jshtml())



In [ ]:
dataset_name = "sim_results_my_behavior_seed_0.pkl"
df_run0 = load_run_dataframe(dataset_name)

anim, html = visualize_simulation_partial(df_run0, start_time_str="9:00", duration_minutes=5, highlight_bee_ids=[3, 5, 14])
html

In [ ]:
# Saving the animation to HTML file
with open("simulation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("simulation_animation.html")